### Humanoid

successor of Robot


In [52]:
import threading
from enum import Enum, auto
from time import sleep
from random import randint
from math import isclose, cos, sin, radians
from typing import Dict, List

In [53]:
# Status :Enum

class Status(Enum):
    # belongs in the battery class but typing won't allow  self.Status as a type
    CHARGING = auto()
    NOT_CHARGING = auto()

    # battery health
    GOOD = auto()

    # humanoid Status will move them elsewhere
    ON = auto()

In [54]:
# Battery


class Battery:
    def __init__(self, percentage: int = 0):
        self.percentage: float = percentage
        self.status: Status = Status.NOT_CHARGING
        self.charging_time: int = 60  # in seconds

        self.health: Status = (
            Status.GOOD
        )  # "good"|"bad" lol weather or not, to slow the charging (add inaccuracies to charging time)

    def _mod_battery(self, percentage: int = 1) -> bool:
        if (self.percentage + percentage) in range(0, 100 + 1):
            self.percentage =+ percentage
            self.status = Status.NOT_CHARGING
            return True
        return False

    def charge(self, charge_length: int) -> bool:
        # charge_length: how long has it been charging or do you want to charge

        # really would like to instead have a method connect-charger that just runs in the bg and and ticks at time intervals updating percentage if still charging stops when t.cancel() is called or something closer to actual charging
        self.status = Status.CHARGING
        timer = threading.Timer(
            interval=charge_length,
            function=self._mod_battery,
            kwargs={"percentage": (charge_length / self.charging_time) * 100},
        )
        # guess a better way would be to say how long a full charge takes then define a function to map the appropriate percentage based on the interval the length of our current timer (lol bette)
        timer.start()
        return True

    def use_up(self, cost: float):
        # based on how it goes was thinking of using time to use up battery as the action is being performed at the rate of cost
        for _ in range(int(cost)):
            if self.percentage - 1 < 0:
                return False   # assumes you act until battery is dead
            else:
                self.percentage -= 1
        return True

    def __str__(self):
        return f"{self.status} :{self.percentage}"

In [55]:
# test : battery charging and use

me = Battery()
print(me.status, " ; ", me.percentage)  # initial

me.charge(3)
print(me.status, " ; ", me.percentage)  # action

sleep(4)
print(me.status, " ; ", me.percentage)  # results

me.use_up(5)
print(me.status, " ; ", me.percentage)  # results 1

me.use_up(5)

Status.NOT_CHARGING  ;  0
Status.CHARGING  ;  0
Status.NOT_CHARGING  ;  5.0
Status.NOT_CHARGING  ;  0.0


False

In [56]:
# Angle

class Angle:
    def __init__(self, theta:float=0):
        self.theta = theta

    @property
    def theta(self):
        return self._theta

    @theta.setter
    def theta(self, theta:float):
        self._theta = theta % 360

    def in_radians(self):
        return radians(self.theta)

    def _turn(self, beta: float =90, is_clockwise: bool=True):
        self.theta = (self.theta + (1 if is_clockwise else -1) * beta) % 360
        return True

    def clockwise(self, beta: float= 90):
        return self._turn(beta, True)

    def anticlockwise(self, beta:float = 90):
        return self._turn(beta, False)

In [57]:
# Test : Angle turn test

a = Angle()

print(a.theta)

# print(a.clockwise())
# print(a.theta)
# print(a.clockwise())
# print(a.theta)
# print(a.clockwise())
# print(a.theta)
# print(a.clockwise())
# print(a.theta)
# print(a.clockwise())
# print(a.theta)

# print(a.anticlockwise())
# print(a.theta)
# print(a.anticlockwise())
# print(a.theta)
# print(a.anticlockwise())
# print(a.theta)
# print(a.anticlockwise())
# print(a.theta)
# print(a.anticlockwise())
# print(a.theta)

# Test : theta assignment

a.theta = 90
print(a.theta)

a.theta = 360
print(a.theta)

a.theta = 361
print(a.theta)

a = Angle(0)
print(a.theta)

a = Angle(361)
print(a.theta)


0
90
0
1
0
1


In [64]:
# Coord

class Coord:
    def __init__(self, x: float = 0, y: float = 0):
        self.x: float = x
        self.y: float = y

    @property
    def x(self):
        return self._x

    @property
    def y(self):
        return self._y
    
    @x.setter
    def x(self, a:float):
        self._x = round(a, 2)
        # rounded, potentially fatal if not corrected
    
    @y.setter
    def y(self, b:float):
        self._y = round(b, 2)
        # rounded

    def __eq__(self, other: object) -> bool:
        """Check equality of two coordinates."""
        if not isinstance(other, Coord):
            return NotImplemented
        return isclose(self.x, other.x, abs_tol=1e-9) and isclose(
            self.y, other.y, abs_tol=1e-9
        )

    def __hash__(self) -> int:
        """Return a hash of the angle for hashable collections."""
        return hash((self.x, self.y))

    def __repr__(self):
        return f"Coord(x={self.x}, y={self.y})"

In [59]:
# Point

class Point:
    def __init__(self, x:float, y:float, theta:float):
        self._angle:Angle = Angle(theta)
        self._coord:Coord = Coord(x, y)

    @property
    def theta(self):
        return self._angle.theta

    @theta.setter
    def theta(self, beta:float):
        self._angle.theta = beta

    @property
    def x(self):
        return self._coord.x

    @property
    def y(self):
        return self._coord.y

    @x.setter
    def x(self, a:float):
        self._coord.x = a
    @y.setter
    def y(self, b:float):   
        self._coord.y = b

    # add isclose for equality check

    def clockwise(self, beta: float = 90):
        return self._angle.clockwise(beta)

    def anticlockwise(self, beta: float = 90):
        return self._angle.anticlockwise(beta)

In [60]:
# Test : Point

p = Point(0, 0, 361)

print(p.theta)
print(p.x)
print(p.y)

p.theta = 362
print(p.theta)

p.x = 10
p.y = 10

print(p.x)
print(p.y)



1
0
0
2
10
10


In [75]:
# body


class Body:
    def __init__(self, max_length: int = 2):
        self.head: Point = Point(0, 0, 360)
        self.tail: Point = self.head
        
        self._parts: List[Point] = [self.head]   # initially the body is just a head, but as it moves it will grow to a max length of 2
        self._max_length: int = max_length

    @property
    def direction(self):
        return self.head.theta

    @property
    def length(self):
        return len(self._parts)

    @property
    def length_max(self):
        return self._max_length

    @length_max.setter
    def length_max(self, new_max: int):
        self._max_length = new_max    # fix the naming un intuitive

    def move(self, length: float = 1):
        new: Point = Point(
            self.head.x + length * cos(radians(self.head.theta)),
            self.head.y + length * sin(radians(self.head.theta)),
            self.head.theta,
        )

        self._parts.insert(0, new)   # slider to a new point, prev moves towards tail

        if len(self._parts) > self._max_length: self._parts.pop()  # remove oldest tail

        self.head, self.tail = self._parts[0], self._parts[-1]  # update references

        # not done, something about the world boundaries, obstacles and collisions
        return True

    def clockwise(self, beta: float = 90):
        return self.head.clockwise(beta)

    def anticlockwise(self, beta: float = 90) -> bool:
        return self.head.anticlockwise(beta)

In [73]:
# Test : moving body

b = Body()

print(f"looking = {b.direction}")
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n')
print("move")
b.move()
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n')
print("move")
b.move()
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n')
print("move")
b.move()
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n')
print("move")
b.move()
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y)



looking = 0
0 0
0 0

move
1.0 0.0
0 0

move
2.0 0.0
1.0 0.0

move
3.0 0.0
2.0 0.0

move
4.0 0.0
3.0 0.0


In [70]:
# test : turning body and moving

b = Body()

print(f"looking = {b.direction}")
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n')

b.clockwise()
print("turn clockwise now looking = ", b.direction)
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n')

print("move")
b.move()
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n')

b.move()
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n')

b.move()
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n')

b.move()
print(b.head.x, b.head.y)
print(b.tail.x, b.tail.y, end='\n\n')


looking = 0
0 0
0 0

turn clockwise now looking =  90
0 0
0 0

move
0.0 1.0
0 0

0.0 2.0
0.0 1.0

0.0 3.0
0.0 2.0

0.0 4.0
0.0 3.0



In [85]:
b = Body()

print(f"current length = {b.length}")
print(f"max length = {b.length_max}\n")

b.move()
print(f"current length = {b.length}")
print(f"max length = {b.length_max}\n")

b.move()
print(f"current length = {b.length}")
print(f"max length = {b.length_max}\n")

b.length_max = 3
print(f"current length = {b.length}")
print(f"max length = {b.length_max}\n")

b.move()
print(f"current length = {b.length}")
print(f"max length = {b.length_max}")

current length = 1
max length = 2

current length = 2
max length = 2

current length = 2
max length = 2

current length = 2
max length = 3

current length = 3
max length = 3


In [ ]:
# Humanoid

class Humanoid:
    def __init__(self):
        self.status: Status = Status.ON
        self._battery: Battery = Battery()
        self._body: Body = Body()

    @property
    def battery_percentage(self):
        return self._battery.percentage

    @property
    def direction(self):
        return self._body.direction

    def charge_battery(self, charge_length: int):
        self.status = Status.CHARGING
        result = self._battery.charge(charge_length)
        return result

    def move(self, length: float = 1):
        self._battery.use_up(length * 0.1 )    # arbitrary cost per unit length (a percentage)
        return self._body.move(length)    # move and use up battery, must interact with the world, check for collisions, etc.

    def clockwise(self, beta: float = 90):
        self._battery.use_up(0.1)
        return self._body.clockwise(beta)

    def anticlockwise(self, beta: float = 90):
        self._battery.use_up(0.1)  # arbitrary cost for turning (a percentage)
        return self._body.anticlockwise(beta)

    def __str__(self):
        return f"{self.status} : {self.battery_percentage:.2f}%"

    def __repr__(self):
        return f"Humanoid({self._battery})"

In [ ]:
# test : humanoid

me = Humanoid()

# print(me.battery_percentage, me.status, sep=" : ")  # initial

# me.charge_battery(3)
# print(me.battery_percentage, me.status, sep=" : ")  # action

# sleep(4)
# print(me.battery_percentage, me.status, sep=" : ")  # results

print(me.battery_percentage, me.direction)
me.clockwise()
print(me.battery_percentage, me.direction)
me.anticlockwise(45)
print(me.battery_percentage, me.direction)

In [ ]:
# world

class World:
    def __init__(self):
        # valid coords x : -50 -> 50 ; y : -50 -> 50
        self.x_range = range(-5, 5+1)
        self.y_range = range(-5, 5+1)

        self.humanoids: Dict[Coord, Humanoid] = {}

    def spawn_humanoid(self):
        coord = Coord(randint(self.x_range.start, self.x_range.stop-1), y=randint(self.y_range.start, self.y_range.stop-1))

        if self.humanoids.get(coord) is None:
            self.humanoids[coord] = Humanoid()
            return True
        return False

In [ ]:
# test : world spawn

w = World()

print(w.humanoids)  # initial

print(w.spawn_humanoid())  # action
print(w.humanoids)
